# YOLOv8 — experimentos com lombadas
Selecione **Ambiente de execução → Alterar tipo → GPU**. Revise primeiro os casos em `data/annotation_review/index.html` no computador. Envie `data/colab_bundle.zip` gerado pelo projeto. Esta sessão prepara um Python 3.13 isolado com as versões do uv.lock. A instalação e o download dos pesos exigem internet. Nenhum comando avalia o teste.
Os resultados serão gravados no seu Google Drive. Cada experimento recusa sobrescrever um diretório existente. Para repetir, use outra pasta de saída. A disponibilidade da GPU e o tempo da sessão dependem do Colab.

In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
assert "colab_bundle.zip" in uploaded, "Selecione colab_bundle.zip"
project = Path("/content/what-is-the-book-2")
project.mkdir(exist_ok=True)
with zipfile.ZipFile("colab_bundle.zip") as archive:
    assert all((project / name).resolve().is_relative_to(project) for name in archive.namelist())
    archive.extractall(project)
del uploaded
%cd /content/what-is-the-book-2

In [ ]:
import subprocess

subprocess.run(["pip", "install", "uv"], check=True)
subprocess.run(
    ["uv", "sync", "--locked", "--python", "3.13", "--extra", "training", "--no-dev"], check=True
)
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-c",
        "import torch; print(torch.__version__, torch.version.cuda); "
        "assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))",
    ],
    check=True,
)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
output = "/content/drive/MyDrive/what-is-the-book-2/runs"
base = ["uv", "run", "--no-sync", "python", "src/detection/train_grid.py", "--output", output]
subprocess.run(base, check=True)  # Apenas apresenta as oito combinações

## Verificação curta
Uma época com 5% do treino. Serve para validar ambiente e memória; não é resultado científico. Se houver falta de memória, reduza `batch` em configs/yolov8_grid.json e use o mesmo valor em todos os experimentos completos.

In [ ]:
subprocess.run(base + ["--run", "--smoke", "--experiment", "1"], check=True)

## Experimentos completos
A próxima célula executa um experimento por vez. Altere EXPERIMENT de 1 a 8. Para executar toda a grade em uma sessão, remova `--experiment` e seu valor. Avalie o tempo medido antes de iniciar toda a grade. Resultados ficam em `runs/grid/exp_XX/`; ranking.json compara apenas execuções concluídas pela mAP50–95 de validação. Se uma execução for interrompida, preserve o diretório e use uma nova pasta para refazer; retomada automática ainda não foi implementada.

In [ ]:
EXPERIMENT = 1
subprocess.run(base + ["--run", "--experiment", str(EXPERIMENT)], check=True)